<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading

**Chapter 06 &mdash; Event-Based Backtesting**

## Applying the Classes

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_core.git
import sys
sys.path.append('python_for_algo_trading_core')


In [ ]:
%run BacktestBase.py

In [ ]:
%run BacktestLongOnly.py

In [ ]:
%run BacktestLongShort.py

## Financial Data Class

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
url = 'http://hilpisch.com/pyalgo_eikon_eod_data.csv'

In [ ]:
raw = pd.read_csv(url, index_col=0, parse_dates=True).dropna()

In [ ]:
raw.info()

In [ ]:
class FinancialData:
    def __init__(self, symbol):
        self.symbol = symbol
        self.retrieve_data()
        self.prepare_data()
    def retrieve_data(self):
        self.raw = pd.read_csv(url, index_col=0, parse_dates=True).dropna()
    def prepare_data(self):
        self.data = pd.DataFrame(self.raw[self.symbol])
        self.data['r'] = np.log(self.data / self.data.shift(1))
    def plot_data(self, cols=None):
        if cols is None:
            cols = self.symbol
        self.data[cols].plot(figsize=(10, 6), title=f'{self.symbol}')

In [ ]:
fd = FinancialData('EUR=')

In [ ]:
fd.raw.head()

In [ ]:
fd.data.head()

In [ ]:
fd.plot_data()

## Event-Based View

... or thinking in **bars**.

In [ ]:
for i in range(10):
    print(i)

In [ ]:
import time
import random

In [ ]:
for bar in range(10):
    print(bar)
    time.sleep(random.random() * 2)

In [ ]:
fd.data.head()

In [ ]:
for bar in range(10):
    print(bar, str(fd.data.index[bar])[:10], fd.data[fd.symbol].iloc[bar])
    time.sleep(random.random() * 2)

In [ ]:
int(5000 / 1.4368)

## Backtesting Base Class

In [ ]:
class BacktestBase(FinancialData):
    def __init__(self, symbol, amount, verbose=False):
        super(BacktestBase, self).__init__(symbol)
        self.initial_balance = amount
        self.current_balance = amount
        self.units = 0
        self.trades = 0
        self.verbose= verbose
    def get_date_price(self, bar):
        date = str(self.data.index[bar])[:10]
        price = self.data[self.symbol].iloc[bar]
        return date, price
    def print_current_balance(self, bar):
        date, price = self.get_date_price(bar)
        print(f'{date} | current balance = {self.current_balance:.2f}')
    def print_net_wealth(self, bar):
        date, price = self.get_date_price(bar)
        net_wealth = self.current_balance + self.units * price
        print(f'{date} | net wealth = {net_wealth:.2f}')
    def place_buy_order(self, bar, units=None, amount=None):
        date, price = self.get_date_price(bar)
        if amount is not None:
            units = int(amount / price)
        self.current_balance -= units * price
        self.units += units
        self.trades += 1
        if self.verbose:
            print(f'{date} | buying {units} for {price}')
    def place_sell_order(self, bar, units=None, amount=None):
        date, price = self.get_date_price(bar)
        if amount is not None:
            units = int(amount / price)
        self.current_balance += units * price
        self.units -= units
        self.trades += 1
        if self.verbose:
            print(f'{date} | selling {units} for {price}')
    def close_out(self, bar):
        date, price = self.get_date_price(bar)
        print(58 * '=')
        print(f'{date} | *** CLOSING OUT FINAL POSITION ***')
        self.current_balance += self.units * price
        print(f'{date} | closing out position of {self.units} for {price}')
        self.units = 0
        self.trades += 1
        perf = (self.current_balance - self.initial_balance) / self.initial_balance * 100
        self.print_current_balance(bar)
        print(f'{date} | net performance [%] = {perf:.2f}')
        print(f'{date} | # of trades executed = {self.trades}')
        print(58 * '=')

In [ ]:
bb = BacktestBase('EUR=', 10000)

In [ ]:
# bb.raw.head()

In [ ]:
bb.get_date_price(100)

In [ ]:
bb.print_current_balance(100)

In [ ]:
bb.print_net_wealth(100)

In [ ]:
bb.place_buy_order(105, units=100)

In [ ]:
bb.print_current_balance(105)

In [ ]:
bb.print_net_wealth(105)

In [ ]:
bb.print_net_wealth(150)

In [ ]:
bb.place_buy_order(200, amount=1000)

In [ ]:
bb.print_current_balance(200)

In [ ]:
bb.print_net_wealth(200)

In [ ]:
bb.units

In [ ]:
bb.place_sell_order(300, units=228)

In [ ]:
bb.print_current_balance(300)

In [ ]:
bb.print_net_wealth(300)

In [ ]:
bb.units

In [ ]:
bb.close_out(500)

## Long Short Strategy Class

In [ ]:
class SMALongShort(BacktestBase):
    
    def prepare_statistics(self):
        self.data['SMA1'] = self.data[self.symbol].rolling(self.SMA1).mean()
        self.data['SMA2'] = self.data[self.symbol].rolling(self.SMA2).mean()
    
    def run_strategy(self, SMA1, SMA2):
        self.SMA1 = SMA1
        self.SMA2 = SMA2
        self.prepare_statistics()
        self.position = 0
        self.trades = 0
        self.units = 0
        self.current_balance = self.initial_balance
        print(58 * '=')
        print(f'*** BACKTESTING STRATEGY ***')
        print(f'{self.symbol} | SMA1={self.SMA1} | SMA2={self.SMA2}')
        print(58 * '=')
        for bar in range(self.SMA2 - 1, len(self.data) - 1):
            trade = False
            if self.position in [0, -1]:
                if self.data['SMA1'].iloc[bar] > self.data['SMA2'].iloc[bar]:
                    self.place_buy_order(bar, units=(1 - self.position) * 5000)
                    self.position = 1
                    trade = True
            elif self.position in [0, 1]:
                if self.data['SMA1'].iloc[bar] < self.data['SMA2'].iloc[bar]:
                    self.place_sell_order(bar, units=(1 + self.position) * 5000)
                    self.position = -1
                    trade = True
            if trade and self.verbose:
                self.print_current_balance(bar)
                self.print_net_wealth(bar)
                print(58 * '-')
        self.close_out(bar)

In [ ]:
sma = SMALongShort('EUR=', 10000, verbose=False)

In [ ]:
sma.run_strategy(30, 180)

In [ ]:
list(zip([30, 30, 42, 42], [180, 252, 180, 252]))

In [ ]:
for SMA1, SMA2 in zip([30, 30, 42, 42], [180, 252, 180, 252]):
    sma.run_strategy(SMA1, SMA2)

In [ ]:
from itertools import product

In [ ]:
list(product(range(30, 51, 10), range(180, 261, 20)))

In [ ]:
for SMA1, SMA2 in product(range(30, 51, 10), range(180, 261, 20)):
    sma.run_strategy(SMA1, SMA2)

In [ ]:
sma.data[252-3:252+3]

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>